# Training an LSTM, and Reading Its Limits

**Computational Text Analysis Pack · Module 24 — Sequence Models**

*Technique:* Keras LSTM text classification; comparison against the bag-of-words baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/12-neural-foundations/24b_lstm_classification.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Build and train an LSTM, then hold it to the same standard as everything else in
this course: does it beat the cheap baseline, on the same data, with the same
split?

## The tooling

**Keras** (on TensorFlow) for the models here — its `Sequential` API is the most
readable way to state an architecture, which is why teaching material reaches
for it. **PyTorch** is the research default and what almost every paper you read
will use; the concepts transfer directly, only the syntax differs.

Both are installed here. Nothing in these two modules needs a GPU: the models
are a few hundred thousand parameters and train on CPU in under a minute.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from tensorflow.keras.layers import TextVectorization

sub = df[df["n_words"] >= 30].sample(n=4000, random_state=1).reset_index(drop=True)
train_df, test_df = train_test_split(sub, test_size=0.25, random_state=0,
                                     stratify=sub["category"])
MAX_WORDS, MAX_LEN, DIM = 8000, 150, 32
vec = TextVectorization(max_tokens=MAX_WORDS, output_sequence_length=MAX_LEN)
vec.adapt(train_df["text"].values)
x_train = vec(train_df["text"].values).numpy()
x_test = vec(test_df["text"].values).numpy()
lb = LabelBinarizer()
y_train, y_test = lb.fit_transform(train_df["category"]), lb.transform(test_df["category"])
print("x_train", x_train.shape, "| classes", [str(c) for c in lb.classes_])

In [ ]:
from tensorflow.keras import Input, Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, GlobalAveragePooling1D
import time

def build(kind):
    layers = [Input(shape=(MAX_LEN,), dtype="int32"),
              Embedding(input_dim=MAX_WORDS, output_dim=DIM)]
    layers += ([LSTM(48)] if kind == "lstm" else [GlobalAveragePooling1D()])
    layers += [Dense(len(lb.classes_), activation="softmax")]
    m = Sequential(layers)
    m.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return m

results = {}
for kind in ["average", "lstm"]:
    m = build(kind)
    t0 = time.time()
    h = m.fit(x_train, y_train, validation_split=0.2, epochs=6, verbose=0)
    _, acc = m.evaluate(x_test, y_test, verbose=0)
    results[kind] = {"acc": acc, "secs": time.time() - t0,
                     "params": m.count_params(), "history": h.history}
    print(f"{kind:<10}{m.count_params():>10,} params  "
          f"{results[kind]['secs']:>6.0f}s  test acc {acc:.3f}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

t0 = time.time()
tv = TfidfVectorizer(token_pattern=r"\w+", min_df=3, max_features=20000, sublinear_tf=True)
A = tv.fit_transform(train_df["text"]); B = tv.transform(test_df["text"])
lr = LogisticRegression(max_iter=1000).fit(A, train_df["category"])
lr_acc = accuracy_score(test_df["category"], lr.predict(B))
lr_secs = time.time() - t0

print(f"\n{'model':<26}{'params':>12}{'seconds':>10}{'accuracy':>11}")
print("-" * 60)
print(f"{'TF-IDF + logistic':<26}{'~20k weights':>12}{lr_secs:>10.0f}{lr_acc:>11.3f}")
print(f"{'Embedding-average':<26}{results['average']['params']:>12,}"
      f"{results['average']['secs']:>10.0f}{results['average']['acc']:>11.3f}")
print(f"{'LSTM':<26}{results['lstm']['params']:>12,}"
      f"{results['lstm']['secs']:>10.0f}{results['lstm']['acc']:>11.3f}")
print()
print("Compare the seconds column against the accuracy column before deciding")
print("the LSTM was worth it.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for kind, style in [("average", "-o"), ("lstm", "-s")]:
    ax.plot(results[kind]["history"]["val_accuracy"], style, label=f"{kind} (val)")
ax.axhline(lr_acc, ls="--", color="grey", label="TF-IDF + logistic")
ax.set_xlabel("epoch"); ax.set_ylabel("accuracy"); ax.legend()
ax.set_title("Does the sequence model overtake the baseline?")
plt.tight_layout()

## Why the field moved on from LSTMs

Three limits, all visible in what you just ran.

1. **Sequential computation.** The LSTM must read token 1 before token 2, so it
   cannot be parallelised across the sequence. That is most of the time
   difference above.
2. **Long-distance decay.** Information from early tokens degrades by the end,
   which is why the gap over the averaging model is small on documents.
3. **One direction, one state.** Everything must be squeezed into a single
   fixed-size vector.

Attention fixes all three at once, which is the next module.

### 🔧 Your turn 1

Cut MAX_LEN from 150 to 40 and re-run. Which model suffers most?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The averaging model usually suffers least, because it was only ever counting
words and a truncated document still contains most of the topical vocabulary.
The LSTM suffers more when the discriminating material sits late in the
document, and it can also *improve*, because shorter sequences mean less state
decay. That instability under a preprocessing change is itself a reason to be
sceptical of small reported gains from sequence models on document tasks.

</details>